# Comparación entre etapas

Lee `Resultados/etapa_*/metricas.json` y compara accuracy entre etapas, para las arquitecturas
que tengan el mismo nombre en todas (`cnn`, `lstm`, `gru`, `tcn`, `ulcnn`). No entrena ni
modifica nada.

Por defecto compara las etapas armadas desde el sweep (0 a 3). `etapa_0_original/` y
`etapa_1_augmentation_curriculum/` quedaron de un camino anterior (snapshot de los notebooks,
con otro esquema de `metricas.json`) y no entran por default — agregalas a mano a `ETAPAS` si
las necesitás, pero no son directamente comparables (otra receta, otro origen).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTADOS = Path(".") if Path("sweep").exists() else Path("Resultados")

# Etapas a comparar, en el orden en que se quieren mostrar.
ETAPAS = [
    "etapa_0_baseline",             # sin augmentation, sin scheduler, sin curriculum
    "etapa_1_augmentation_scheduler",  # + augmentation + scheduler
    "etapa_2_curriculum",          # + curriculum (datos RadioML reales)
    "etapa_3_sintetico_curriculum",  # datos sinteticos + curriculum (test sigue siendo RadioML real)
]

rows = []
for etapa in ETAPAS:
    d = json.loads((RESULTADOS / etapa / "metricas.json").read_text(encoding="utf-8"))
    for arch, m in d["modelos"].items():
        arch = arch.rsplit("/", 1)[-1]  # por si viene como "Modelos iniciales/cnn"
        rows.append(dict(
            etapa=etapa, arch=arch, params=m.get("parametros"),
            acc=m.get("accuracy_global"), acc_low=m.get("accuracy_snr_menor_igual_-10"),
            acc_high=m.get("accuracy_snr_mayor_0"), data_source=m.get("data_source", "radioml"),
            curriculum=m.get("curriculum"), por_snr=m.get("accuracy_por_snr"),
        ))
df = pd.DataFrame(rows)
df[["etapa", "arch", "params", "acc", "acc_low", "data_source", "curriculum"]]

In [ ]:
# Accuracy global y a SNR <= -10 dB, una arquitectura al lado de la otra, agrupadas por etapa
archs = sorted(df.arch.unique())
x = range(len(ETAPAS))
ancho = 0.8 / len(archs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, col, titulo in zip(axes, ["acc", "acc_low"], ["Accuracy global", "Accuracy a SNR <= -10 dB"]):
    for i, arch in enumerate(archs):
        g = df[df.arch == arch].set_index("etapa").reindex(ETAPAS)
        ax.bar([xi + i * ancho for xi in x], g[col], width=ancho, label=arch)
    ax.set_xticks([xi + ancho * (len(archs) - 1) / 2 for xi in x])
    ax.set_xticklabels(ETAPAS, rotation=30, ha="right")
    ax.set_title(titulo)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla pivote: una fila por arquitectura, una columna por etapa
pivote = df.pivot(index="arch", columns="etapa", values="acc")[ETAPAS]
pivote_low = df.pivot(index="arch", columns="etapa", values="acc_low")[ETAPAS]
print("Accuracy global\n", pivote.round(3))
print("\nAccuracy a SNR <= -10 dB\n", pivote_low.round(3))

In [ ]:
# Accuracy por SNR, una arquitectura elegida, comparando las etapas
ARCH = "tcn"  # cambiar por cnn, lstm, gru, ulcnn

fig, ax = plt.subplots(figsize=(9, 5))
for etapa in ETAPAS:
    fila = df[(df.arch == ARCH) & (df.etapa == etapa)]
    if fila.empty or fila.iloc[0]["por_snr"] is None:
        continue
    por_snr = fila.iloc[0]["por_snr"]
    niveles = sorted(por_snr, key=int)
    ax.plot([int(s) for s in niveles], [por_snr[s] for s in niveles], marker="o", label=etapa)
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Accuracy")
ax.set_title(f"Accuracy por SNR — {ARCH}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()